# [NASLOV PROJEKTA]

**Seminarski rad iz predmeta Mašinsko učenje**

Ime i prezime · broj indeksa · datum

---

> **Uputstvo za korišćenje ovog šablona**
>
> Poglavlja odgovaraju **tačkama iz pravila ispita**, jedan na jedan. Svako poglavlje ima
> kratko objašnjenje šta se traži i `TODO` ćelije koje treba popuniti.
>
> Kada završiš, **obriši sve `TODO` komentare i ovaj okvir**.
>
> Gotovi isečci koda za svaku fazu su u `biblioteka_koda.ipynb`.

## Pravila ispita

Projekat treba da obuhvata:

| # | zahtev | poglavlje |
|---|---|---|
| 1 | odabir odgovarajućeg dataset-a | **2** |
| 2 | definisanje problema | **1** |
| 3 | preprocesiranje podataka | **4** |
| 4 | treniranje više modela mašinskog učenja po izboru | **5** |
| 5 | evaluaciju i poređenje rezultata modela | **6** |
| 6 | interpretaciju modela i dobijenih rezultata | **7** |

---
# 1. Definisanje problema

**Šta se traži:** jasno reći **šta se predviđa**, iz **čega**, i **kako se meri uspeh**.

Najkraći način da se to uradi je formulacija po Mitchell-u (predavanje 01).

> **TODO** — popuni tabelu i dva pasusa ispod.

| | |
|---|---|
| **Zadatak (Z)** | *šta model treba da uradi* |
| **Iskustvo (I)** | *koji podaci, koliko uzoraka* |
| **Performanse (P)** | *koja metrika i zašto baš ona* |

**Tip problema:** *klasifikacija / regresija / klasterovanje*

**Zašto je ovo problem mašinskog učenja**

*Objasni zašto se pravilo ne može napisati eksplicitno, nego se mora naučiti iz podataka.
Ako se problem može rešiti jednim `if`-om, nije dobra tema.*

**Zašto je problem zanimljiv**

*Kome bi rezultat koristio i šta bi se s njim moglo uraditi.*

---
# 2. Odabir dataset-a

**Šta se traži:** opisati odakle podaci dolaze, koliko ih ima i šta predstavljaju.

> **TODO** — popuni opis i tabelu obeležja.

**Izvor:** *link ili opis (Kaggle, UCI, sopstveno prikupljanje, API...)*

**Zašto baš ovaj skup:** *veza sa problemom iz poglavlja 1*

In [ ]:
# TODO: podesi importe prema onome sto koristis
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
RS = 42                      # jedan seed kroz ceo rad — zbog ponovljivosti
np.random.seed(RS)

In [ ]:
# TODO: ucitaj svoj skup
# df = pd.read_csv("data/tvoj_skup.csv")

print(f"Oblik: {df.shape[0]} redova x {df.shape[1]} kolona")
df.head()

In [ ]:
# TODO: opis obelezja — sta koja kolona znaci
print(df.info())
print("\nOsnovna statistika:")
df.describe().T

---
# 3. Istraživačka analiza podataka (EDA)

**Nije eksplicitno u pravilima, ali se podrazumeva** — bez nje se ne može opravdati nijedna
odluka u preprocesiranju.

Obavezno pokazati:

- **raspodelu ciljne promenljive** — ovo određuje koje metrike smeš da koristiš
- nedostajuće vrednosti
- korelacije između obeležja
- 2–3 grafika koja nešto **stvarno pokazuju**, ne ukras

> **TODO** — dopuni analizu prema svom skupu.

In [ ]:
# TODO: zameni "ciljna" imenom svoje ciljne kolone
CILJ = "ciljna"

print("Raspodela ciljne promenljive:")
print(df[CILJ].value_counts())
print(f"\nUdeo manjinske klase: {df[CILJ].value_counts(normalize=True).min():.1%}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df[CILJ].value_counts().plot.bar(ax=ax[0], rot=0, color="#4C72B0")
ax[0].set_title("Raspodela klasa")
df.isna().sum().sort_values(ascending=False).head(10).plot.barh(ax=ax[1], color="#C44E52")
ax[1].set_title("Nedostajuce vrednosti (10 najgorih)")
plt.tight_layout(); plt.show()

In [ ]:
# korelaciona matrica
num = df.select_dtypes(include=[np.number])
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(num.corr(), cmap="RdBu_r", center=0, square=True,
            linewidths=.5, cbar_kws={"shrink": .7}, ax=ax)
ax.set_title("Korelaciona matrica")
plt.tight_layout(); plt.show()

### Zaključci iz EDA

> **TODO** — u 3–5 rečenica: šta si video i **kako to utiče na naredne korake**.
> Npr. „klase su neuravnotežene 90:10, pa tačnost nije upotrebljiva metrika" ili
> „tri obeležja su međusobno korelisana preko 0.9, pa razmatram PCA".

---
# 4. Preprocesiranje podataka

**Šta se traži:** pripremiti podatke i **obrazložiti svaku odluku**.

Tipični koraci:

| korak | kada je potreban |
|---|---|
| **podela na train/test** | uvek, **pre** svega ostalog |
| nedostajuće vrednosti | ako ih ima |
| kodiranje kategoričkih | ako ima tekstualnih kolona (`get_dummies`, `OneHotEncoder`) |
| **skaliranje** | obavezno za kNN, SVM, neuronske mreže, PCA |
| balansiranje klasa | ako su klase jako neuravnotežene |

> **Najvažnije pravilo:** sve što se **uči** iz podataka (skaler, imputer, PCA, kodiranje)
> mora se naučiti **samo na trening skupu**, pa primeniti na test.
> Najsigurnije je koristiti `Pipeline`.

> **TODO** — sprovedi korake koje tvoj skup traži i **napiši zašto**.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=[CILJ])
y = df[CILJ]

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.2, random_state=RS,
    stratify=y            # TODO: izbaci ako je regresija
)
print(f"Trening: {Xtr.shape[0]} uzoraka")
print(f"Test   : {Xte.shape[0]} uzoraka")
print(f"\nRaspodela klasa — trening: {np.bincount(ytr).tolist()}")
print(f"Raspodela klasa — test   : {np.bincount(yte).tolist()}")

In [ ]:
# TODO: preprocesiranje kroz Pipeline — sam pazi na redosled i sprecava leakage
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

numericke = X.select_dtypes(include=[np.number]).columns.tolist()
kategoricke = X.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Numerickih obelezja  : {len(numericke)}")
print(f"Kategorickih obelezja: {len(kategoricke)}")

priprema = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), numericke),
    ("kat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), kategoricke),
])

### Obrazloženje odluka u preprocesiranju

> **TODO** — za svaki korak napiši **zašto** si ga primenio.
> Npr. „nedostajuće vrednosti popunjene medijanom jer je raspodela iskošena";
> „skaliranje je obavezno jer koristim kNN i SVM koji se oslanjaju na rastojanja".

---
# 5. Treniranje više modela

**Šta se traži:** obučiti **više** modela i uporediti ih.

Preporuka: **5–8 modela**, iz različitih porodica, plus **bazni model**.

| porodica | modeli | predavanje |
|---|---|---|
| bazni | `DummyClassifier` | 07 |
| linearni | logistička / linearna regresija | 02 |
| instance-based | kNN | 03 |
| stabla | `DecisionTree` | 04 |
| probabilistički | `GaussianNB` | 05 |
| margine | SVM | 06 |
| ansambli | RandomForest, XGBoost, Voting, Stacking | 12 |
| duboko učenje | MLP | 10 |

> **Bazni model je obavezan.** Bez njega se ne zna šta je „loše" — 90% tačnosti ne znači
> ništa ako 90% uzoraka pripada jednoj klasi.

> **TODO** — izaberi modele i obuči ih **istim postupkom**, da poređenje bude pošteno.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(5, shuffle=True, random_state=RS)

MODELI = {
    "Dummy (bazni)":        DummyClassifier(strategy="most_frequent"),
    "Logisticka regresija": LogisticRegression(max_iter=3000, random_state=RS),
    "kNN":                  KNeighborsClassifier(),
    "Stablo odlucivanja":   DecisionTreeClassifier(max_depth=5, random_state=RS),
    "Naivni Bajes":         GaussianNB(),
    "SVM (RBF)":            SVC(probability=True, random_state=RS),
    "Random Forest":        RandomForestClassifier(n_estimators=300, random_state=RS, n_jobs=-1),
}
# TODO: dodaj XGBoost / MLP / Voting / Stacking ako zelis

In [ ]:
rezultati = {}
modeli_fit = {}

for ime, model in MODELI.items():
    pipe = Pipeline([("prep", priprema), ("m", model)])
    cv_skor = cross_val_score(pipe, Xtr, ytr, cv=cv, scoring="accuracy", n_jobs=-1)
    pipe.fit(Xtr, ytr)
    modeli_fit[ime] = pipe
    rezultati[ime] = {"cv_prosek": cv_skor.mean(), "cv_std": cv_skor.std()}
    print(f"{ime:<24} CV = {cv_skor.mean():.4f} +- {cv_skor.std():.4f}")

### Podešavanje hiperparametara

> **TODO** — podesi hiperparametre bar jednog modela pomoću `GridSearchCV`, i **prikaži
> koliko je to pomoglo**. Ovo pokriva deo gradiva sa predavanja 07.

In [ ]:
from sklearn.model_selection import GridSearchCV

# TODO: prilagodi mrezu svom modelu
mreza = {
    "m__n_estimators": [100, 300],
    "m__max_depth": [5, 10, None],
    "m__min_samples_leaf": [1, 5],
}
gs = GridSearchCV(Pipeline([("prep", priprema),
                            ("m", RandomForestClassifier(random_state=RS, n_jobs=-1))]),
                  mreza, cv=cv, scoring="accuracy", n_jobs=-1)
gs.fit(Xtr, ytr)

print(f"Najbolji parametri : {gs.best_params_}")
print(f"Najbolji CV rezultat: {gs.best_score_:.4f}")
print(f"Isprobano kombinacija: {len(gs.cv_results_['params'])}")

modeli_fit["Random Forest (podesen)"] = gs.best_estimator_
rezultati["Random Forest (podesen)"] = {"cv_prosek": gs.best_score_, "cv_std": 0.0}

---
# 6. Evaluacija i poređenje rezultata

**Šta se traži:** uporediti modele **istim metrikama** i prikazati rezultate pregledno.

> **Izbor metrike zavisi od raspodele klasa.** Ako su neuravnotežene, tačnost vara —
> koristi F1, balanced accuracy, ROC-AUC ili PR-AUC.

Obavezno prikazati:
- **zbirnu tabelu** svih modela
- **matricu konfuzije** najboljeg modela
- bar jednu **krivu** (ROC ili PR)

> **TODO** — dopuni evaluaciju.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             balanced_accuracy_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve, classification_report)

redovi = []
for ime, pipe in modeli_fit.items():
    pred = pipe.predict(Xte)
    red = {
        "model": ime,
        "CV": round(rezultati[ime]["cv_prosek"], 4),
        "tacnost": round(accuracy_score(yte, pred), 4),
        "balanced_acc": round(balanced_accuracy_score(yte, pred), 4),
        "preciznost": round(precision_score(yte, pred, average="weighted", zero_division=0), 4),
        "odziv": round(recall_score(yte, pred, average="weighted"), 4),
        "F1": round(f1_score(yte, pred, average="weighted"), 4),
    }
    if hasattr(pipe, "predict_proba") and len(np.unique(yte)) == 2:
        red["ROC-AUC"] = round(roc_auc_score(yte, pipe.predict_proba(Xte)[:, 1]), 4)
    redovi.append(red)

tabela = pd.DataFrame(redovi).sort_values("F1", ascending=False).reset_index(drop=True)
display(tabela)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
t = tabela.set_index("model")["F1"].sort_values()
boje = ["#C44E52" if "Dummy" in i else "#4C72B0" for i in t.index]
t.plot.barh(ax=ax, color=boje)
ax.set_xlabel("F1 (weighted)"); ax.set_title("Poredjenje modela")
plt.tight_layout(); plt.show()

In [ ]:
najbolji = tabela.model.iloc[0]
pipe = modeli_fit[najbolji]
pred = pipe.predict(Xte)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay(confusion_matrix(yte, pred)).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Matrica konfuzije — {najbolji}"); ax.grid(False)
plt.tight_layout(); plt.show()

print(f"NAJBOLJI MODEL: {najbolji}\n")
print(classification_report(yte, pred, digits=4))

### Diskusija rezultata

> **TODO** — odgovori na pitanja:
> - Koji model je najbolji i **za koliko** je bolji od baznog?
> - **Zašto** baš taj tip modela radi najbolje na ovim podacima?
> - Da li se neki model ponaša neočekivano loše i zašto? *(npr. naivni Bajes ako su
>   obeležja korelisana — pretpostavka uslovne nezavisnosti je narušena)*
> - Da li je razlika između top modela **stvarna** ili u granicama standardne devijacije CV-a?

---
# 7. Interpretacija modela i rezultata

**Šta se traži:** objasniti **zašto** model donosi odluke koje donosi.

Preporuka — bar dve metode:

| metoda | tip | predavanje |
|---|---|---|
| koeficijenti / `feature_importances_` | model-specifična | 02, 12 |
| **permutaciona važnost** | model-agnostička | 13 |
| **SHAP** | model-agnostička | 13 |
| PDP / ICE | model-agnostička | 13 |

> **TODO** — primeni bar dve metode i **protumači** rezultat rečima.

In [ ]:
from sklearn.inspection import permutation_importance

pi = permutation_importance(pipe, Xte, yte, n_repeats=10,
                            random_state=RS, scoring="f1_weighted", n_jobs=-1)
vazn = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
vazn.head(12)[::-1].plot.barh(ax=ax, color="#4C72B0")
ax.set_xlabel("pad F1 pri mesanju obelezja")
ax.set_title("Permutaciona vaznost")
plt.tight_layout(); plt.show()

In [ ]:
# SHAP — radi najbrze nad modelima zasnovanim na stablima (TreeExplainer)
import shap

# TODO: prilagodi ako tvoj najbolji model nije zasnovan na stablima
model_bez_pipe = pipe.named_steps["m"]
X_prep = pipe.named_steps["prep"].transform(Xte)
if hasattr(X_prep, "toarray"):
    X_prep = X_prep.toarray()

try:
    expl = shap.TreeExplainer(model_bez_pipe)
    sv = expl.shap_values(X_prep)
    if isinstance(sv, list):
        sv = sv[1]
    shap.summary_plot(sv, X_prep, max_display=12, show=False)
    plt.title("SHAP — globalni pregled", pad=18)
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f"TreeExplainer nije primenljiv ({type(e).__name__}).")
    print("Koristi shap.KernelExplainer ili shap.Explainer za druge tipove modela.")

### Tumačenje

> **TODO** — u nekoliko rečenica:
> - Koja obeležja model najviše koristi?
> - Da li to ima **smisla u domenu**? Ako obeležje koje nema veze sa problemom ispadne
>   najvažnije, to je često znak **curenja informacija**.
> - Slažu li se različite metode interpretacije međusobno?
> - **Oprez:** SHAP objašnjava **model**, ne stvarnost. Korelacija nije uzročnost.

---
# 8. Zaključak

> **TODO** — sažmi rad u nekoliko pasusa.

**Šta je urađeno** — problem, podaci, modeli, rezultat u jednoj rečenici.

**Glavni nalazi** — 3–4 tačke; brojevi, ne uopštene tvrdnje.

**Ograničenja** — navedi ih sam, pre nego što ih neko pita:
- veličina i reprezentativnost uzorka
- da li su podaci iz jednog izvora / perioda
- korelacija nije uzročnost
- šta model ne može

**Mogući nastavak** — šta bi sledeće uradio da imaš više vremena ili podataka.